In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
# from helper import load_env
# load_env()
from pydantic import BaseModel, Field
from typing import List, Dict, Type
from typing import List, Optional
import os
import yaml

In [3]:
import os, json, time, gc
import logging 

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, Image, Video
from tqdm import tqdm
from openai import OpenAI, AsyncOpenAI
from openai.types.chat import (ChatCompletion, 
                               ChatCompletionChunk,
                               ChatCompletionContentPartTextParam, 
                               ChatCompletionContentPartImageParam,
                               ChatCompletionStreamOptionsParam)
import asyncio
import aiohttp
import pandas as pd
import re


import base64
from PIL import Image
import io

#fix bug with aysncio and jupyter
import nest_asyncio # for langchain async 
nest_asyncio.apply()

In [4]:
import litellm
from litellm import acompletion, completion

### Test LM Studio Connection By Openai API

In [5]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key= "lm-studio"

In [6]:
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=api_key)

# Replace with the exact model name running in LM Studio
model_name = "qwen3.6-35b-a3b-mtp"  #"google/gemma-4-12b" 

In [ ]:
ret = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": "Explain how to check memory usage in a Jupyter notebook."}
    ],
    temperature=0.7,
)

Markdown(ret.choices[0].message.content)

## Test LM studio LLM Connection by liteLLM API

In [10]:
# Markdown(completion.choices[0].message.content)

In [7]:
# 3. Define the async function
async def get_chat_completion(
    api_base,
    api_key, 
    model_name = "openai/local-model",
    system_prompt= "You are a helpful assistant.",
    user_prompt="",
    temperature= 0.7,
    max_tokens=4096):
    
    response = await acompletion(
        model=model_name,  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
        api_base=api_base,
        api_key=api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response

In [10]:
%%time
# 4. Execute the async function directly in Jupyter
response = asyncio.run(get_chat_completion(api_base=LM_STUDIO_BASE_URL, 
                                           api_key=api_key,
                                           model_name="openai/local-model",
                                            user_prompt="What is LLM?"))



CPU times: user 54.9 ms, sys: 6.07 ms, total: 60.9 ms
Wall time: 17.8 s


In [11]:
Markdown(response.choices[0].message.content)



An **LLM** stands for **Large Language Model**. It's a type of artificial intelligence system designed to understand, generate, and manipulate human language at scale.

### How it works:
LLMs are built on deep learning architectures called **transformers** and trained on massive datasets of text (and sometimes code) scraped from the internet, books, articles, forums, and more. Instead of following explicit programming rules, they learn statistical patterns in language by predicting the next word or "token" in a sequence based on context. This enables them to produce coherent, context-aware responses to prompts.

### Key capabilities:
- Answer questions & hold conversations
- Write essays, emails, stories, scripts, etc.
- Translate languages & summarize long texts
- Generate & debug code
- Perform reasoning tasks (with varying reliability)
- Adapt to new topics with minimal examples ("few-shot" or "zero-shot" learning)

### Common examples:
OpenAI's GPT series, Anthropic's Claude, Google's Gemini, Meta's Llama family, Mistral, and many open-source models. These power AI chatbots, writing assistants, coding tools, research helpers, customer service bots, and more.

### Important limitations:
- **No true understanding**: They're advanced pattern matchers, not conscious or reasoning beings.
- **Hallucinations**: Can confidently generate plausible but factually incorrect information.
- **Bias & safety**: Reflect biases present in training data; require careful alignment and guardrails.
- **Resource-intensive**: Training and running large models demand significant compute power and energy.

In short: LLMs are highly capable AI systems that process and generate human-like text by learning from vast amounts of written data, transforming how we interact with technology across education, work, creativity, and software development. Let me know if you'd like a deeper dive into how they're built or how to use them effectively!

## Generate COT Data

In [8]:
def generate_cot_data(prompt: str, answer: str) -> str:
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.

Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}

Correct Answer: {answer}

Please think step by step inside <think> tags about how to discover the transformation rule."""

    try:
        response = asyncio.run (acompletion(
            model="openai/local-model",  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,
            max_tokens=1600,
            timeout=180
            # reasoning_effort="medium"
        ))
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # === Hardcode the final answer (Most Reliable) ===
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"

        return final_output
        
    except Exception as e:
        print(f"Error generating CoT for prompt: {e}")
        # Fallback: still return something usable
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"
                            

In [9]:
testFile ="../src/Dataset/test.csv"
trainFile = "../src/Dataset/train.csv"
cotFile = "train_cot.csv"


In [10]:
# trainDF = pd.read_csv(trainFile)
# trainDF

In [11]:
# # Add new column for CoT reasoning
# if "cot_reasoning" not in trainDF.columns:
#     trainDF["cot_reasoning"] = ""

In [12]:
# trainDF

In [13]:
outputFile = "train_cot2.csv"               # output file with CoT
# MODEL = "gpt-4o"                         # or "claude-3-5-sonnet-20241022"
DELAY = 0.1                                # seconds between API calls (adjust based on rate limit)

In [14]:
print("Checking for existing train_cot.csv...")

if os.path.exists(outputFile):
    print("Found existing train_cot.csv → Resuming...")
    trainDF = pd.read_csv(outputFile)
else:
    print("No existing file found. Starting from train.csv...")
    trainDF = pd.read_csv(trainFile)
    if "cot_reasoning" not in trainDF.columns:
        trainDF["cot_reasoning"] = ""

# Count how many rows still need processing
remaining = trainDF["cot_reasoning"].isna().sum() + (trainDF["cot_reasoning"] == "").sum()
print(f"Total rows: {len(trainDF)}")
print(f"Rows already processed: {len(trainDF) - remaining}")
print(f"Rows left to process: {remaining}\n")

Checking for existing train_cot.csv...
Found existing train_cot.csv → Resuming...
Total rows: 9500
Rows already processed: 3130
Rows left to process: 6370



In [15]:
trainDF

,id,prompt,answer,cot_reasoning
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,<think>\nHere's a thinking process that leads ...
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,<think>\nThe user wants me to solve a puzzle b...
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book,<think>\nThe user wants me to explain the proc...
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,<think>\nThe user wants me to identify the hid...
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret,<think>\nHere's a thinking process that leads ...
...,...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110,NaN
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45,NaN
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror,NaN
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates,NaN


In [ ]:
%%time
# print("Generating Chain-of-Thought data...")

# for idx in tqdm(range(len(trainDF))):
#     if trainDF.loc[idx, "cot_reasoning"]:   # Skip if already generated
#         continue

#     prompt = trainDF.loc[idx, "prompt"]
#     answer = str(trainDF.loc[idx, "answer"]).strip()

#     cot = generate_cot_data(prompt, answer)
#     trainDF.loc[idx, "cot_reasoning"] = cot

#     # Save progress every 50 rows (in case of crash)
#     if (idx + 1) % 20 == 0:
#         trainDF.to_csv(outputFile, index=False)
#         print(f"Saved progress at row {idx + 1}")

#     time.sleep(DELAY)   # Respect API rate limit

if remaining == 0:
    print("✅ All rows already have CoT reasoning. Nothing to do.")
else:
    print("Starting CoT generation (resume mode)...\n")

    processed_count = 0

    for idx in tqdm(range(len(trainDF))):
        current_cot = trainDF.loc[idx, "cot_reasoning"]

        # Skip if already has content
        if pd.notna(current_cot) and str(current_cot).strip() != "":
            continue

        prompt = trainDF.loc[idx, "prompt"]
        answer = str(trainDF.loc[idx, "answer"]).strip()

        cot = generate_cot_data(prompt, answer)
        trainDF.loc[idx, "cot_reasoning"] = cot
        processed_count += 1

        # Save progress every 50 new rows
        if processed_count % 20 == 0:
            trainDF.to_csv(outputFile, index=False)
            print(f"Saved progress. Processed {processed_count} new rows so far.")

        time.sleep(DELAY)

    # Final save
    trainDF.to_csv(outputFile, index=False)
    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")

Starting CoT generation (resume mode)...



 33%|███████████▉                        | 3151/9500 [16:37<37:26:10, 21.23s/it]

Saved progress. Processed 20 new rows so far.


 33%|████████████                        | 3171/9500 [29:57<78:40:28, 44.75s/it]

Saved progress. Processed 40 new rows so far.


...

 35%|███████████▊                      | 3311/9500 [2:33:06<90:47:56, 52.82s/it]

Saved progress. Processed 180 new rows so far.


 35%|███████████▌                     | 3331/9500 [2:51:44<105:43:46, 61.70s/it]

Saved progress. Processed 200 new rows so far.


 35%|███████████▉                      | 3334/9500 [2:53:52<85:49:17, 50.11s/it]